# Reading, PA — Revenue-Neutral Reassessment (tooling demo)

A walkthrough of `lvt.reassessment`, the toolkit for modeling **tax-base** shifts
(updating stale assessed values and rolling the millage back to revenue neutrality),
as opposed to the **tax-rate** shifts in `lvt.lvt_utils` (split-rate, abatement).

The module is **agnostic to where the new values come from** — you supply a
"new assessed value" column. AVM/LYCD value *generation* lives in the sibling
`berks_open_avmkit` project, not here.

Three pieces:
1. **`model_revenue_neutral_reassessment`** — single jurisdiction, one rolled-back flat rate (Part A).
2. **`model_multi_district_reassessment`** — overlapping taxing districts each rolled back within itself, the PA anti-windfall method (Part B).
3. **`decompose_reassessment_and_lvt`** — separate the reassessment effect from the LVT effect (Part C).

The real Reading decomposition also runs inside `model_lycd.ipynb` Section 7b, which now
calls this same tooling.

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, '../..')

from lvt.reassessment import (
    model_revenue_neutral_reassessment,
    model_multi_district_reassessment,
    decompose_reassessment_and_lvt,
    save_reassessment_export,
)
from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
)

CITY_MILLAGE = 19.217          # City of Reading 2026 adopted flat millage
LAND_IMPROVEMENT_RATIO = 4.0
DATA_DIR = Path('data')
AVM_BASE = Path('C:/projects/berks_open_avmkit/notebooks/pipeline/data/us-pa-berks/out/models')
print('Setup complete.')

## Part A — Single-district reassessment (real Reading data)

Scenario: Berks County reassesses every Reading parcel from its 1994 CAMA value
(`assr_market_value`) to a current AVM market value, and rolls the flat City millage
back so the City levy is unchanged. We supply the AVM total as the new base.

In [ ]:
# Load cached Reading parcels (created by model.ipynb) and the AVM predictions.
parcel_path = DATA_DIR / 'parcels.gpq'
if not parcel_path.exists():
    raise FileNotFoundError(f"Run model.ipynb first to create {parcel_path}.")
gdf = gpd.read_parquet(parcel_path)
for c in ['assr_land_value', 'assr_impr_value', 'assr_market_value']:
    gdf[c] = pd.to_numeric(gdf[c], errors='coerce').fillna(0.0)
gdf['full_exmp'] = (gdf['category_code'] == 'E').astype(int)
gdf['PROPERTY_CATEGORY'] = gdf['category_code']   # demo uses raw class codes as the category

frames = []
for i, mg in enumerate(['residential_sf', 'residential_mf', 'commercial', 'vacant']):
    p = AVM_BASE / mg / 'main' / 'ensemble' / 'pred_universe.parquet'
    if p.exists():
        d = pd.read_parquet(p, columns=['key', 'prediction'])
        d['_pr'] = i
        frames.append(d)
avm = (pd.concat(frames).sort_values('_pr')
       .drop_duplicates('key', keep='first').drop(columns='_pr'))
gdf = gdf.merge(avm, on='key', how='left')

# New base = AVM total; fall back to CAMA scaled to the AVM ratio for uncovered parcels.
ratio = (gdf.loc[gdf['prediction'].notna(), 'prediction'].sum()
         / gdf.loc[gdf['prediction'].notna(), 'assr_market_value'].clip(lower=1).sum())
gdf['avm_value'] = np.where(
    gdf['prediction'].notna(),
    gdf['prediction'].clip(lower=0),
    gdf['assr_market_value'].clip(lower=0) * ratio,
)
print(f"{len(gdf):,} parcels; {gdf['prediction'].notna().mean()*100:.1f}% AVM-covered")

In [ ]:
# One call does the rollback. current_tax is derived from the OLD base x current millage,
# so we don't need to pre-compute it. Exempt parcels (full_exmp=1) drop out of the base.
flat_millage, new_revenue, gdf = model_revenue_neutral_reassessment(
    gdf,
    new_value_col='avm_value',
    old_value_col='assr_market_value',
    current_millage=CITY_MILLAGE,
    exemption_flag_col='full_exmp',
    verbose=True,
)
print()
print(f"Revenue neutral?  current ${gdf['current_tax'].sum():,.0f}  ->  new ${new_revenue:,.0f}")

In [ ]:
# Winners and losers by property class. Under a flat rolled-back rate a parcel wins
# iff its reassessment ratio (new/old) is below the jurisdiction-average ratio.
summary = calculate_category_tax_summary(
    gdf, category_col='category_code',
    current_tax_col='current_tax', new_tax_col='new_tax',
)
print_category_tax_summary(summary, title='Reading — reassessment-only impact by class code')

avg_ratio = gdf['taxable_new_value'].sum() / gdf.loc[gdf['full_exmp'] == 0, 'assr_market_value'].sum()
share_win = (gdf.loc[gdf['full_exmp'] == 0, 'tax_change'] < 0).mean() * 100
print(f"\nJurisdiction-average reassessment ratio: {avg_ratio:.3f}")
print(f"Share of taxable parcels that win (bill falls): {share_win:.1f}%")

In [ ]:
# Export with the reassessment columns appended to the standard schema.
out = save_reassessment_export(
    gdf, 'reading_reassessment_demo', 'data/reading_reassessment_demo.csv',
    model_type='reassessment:flat', land_millage=flat_millage, improvement_millage=flat_millage,
    property_category_col='category_code',
)
print('Exported columns:', list(out.columns))

## Part B — Multi-district (PA anti-windfall)

Pennsylvania's anti-windfall law (53 Pa.C.S. § 8823) requires the rollback to be
revenue-neutral **separately in every taxing district** — the county rolls back over all
its parcels, each municipality within itself, each school district across its whole area.
A parcel's new bill is the sum across the districts it belongs to.

Reading is a single municipality, so this richness only shows up county-wide. We use a
small **synthetic** five-parcel county spanning two municipalities and two school
districts to make the per-district rollback visible. (The real Berks county-wide run
lives in `berks_open_avmkit`.)

In [ ]:
county = pd.DataFrame({
    'parcel': ['p1', 'p2', 'p3', 'p4', 'p5'],
    'old':    [100_000.0, 200_000.0, 150_000.0, 300_000.0, 250_000.0],   # 1994 CAMA
    'new':    [150_000.0, 180_000.0, 300_000.0, 260_000.0, 400_000.0],   # AVM
    'muni':   ['Boro A', 'Boro A', 'Twp B', 'Twp B', 'Twp B'],
    'school': ['SD X', 'SD Y', 'SD X', 'SD Y', 'SD Y'],
}).set_index('parcel')

districts = [
    {'name': 'county', 'id_col': None,     'millage': 9.013},                 # flat, county-wide
    {'name': 'muni',   'id_col': 'muni',   'millage': {'Boro A': 8.0, 'Twp B': 6.0}},
    {'name': 'school', 'id_col': 'school', 'millage': {'SD X': 20.0, 'SD Y': 22.0}},
]

result, district_summary = model_multi_district_reassessment(
    county, new_value_col='new', old_value_col='old', districts=districts, verbose=True,
)
print()
district_summary

In [ ]:
# Each district's new levy equals its old levy (revenue neutral within itself),
# yet parcels move relative to one another. The combined % change is a millage-weighted
# blend of the three per-district reassessment ratios; the function asserts that identity
# internally (crosscheck=True).
cols = ['old', 'new', 'reassessment_ratio', 'current_tax', 'new_tax', 'tax_change', 'tax_change_pct']
result[cols].round(2)

## Part C — Decomposition: reassessment vs LVT

Stacking the two reforms gives three tax points per parcel:

| Point | Basis |
|---|---|
| **current** | old base × current flat rate |
| **reassessed** | new base × rolled-back flat rate (Part A) |
| **reassessed + LVT** | new base split into land/improvement × split-rate |

`decompose_reassessment_and_lvt` splits each parcel's total change into a reassessment
component and an LVT component (`reassess_change + lvt_change == total_change`).

For a quick, self-contained illustration we split the AVM total using the 1994 CAMA land
fraction. The production Reading model uses the LYCD land method instead — see
`model_lycd.ipynb` Section 7b, which calls this same `decompose_reassessment_and_lvt`.

In [ ]:
# Point 2 (reassessed flat) is the new_tax from Part A.
gdf['avm_flat_tax'] = gdf['new_tax']

# Point 3: split-rate on the reassessed base. Split the AVM total with the CAMA land fraction.
land_frac = np.where(gdf['assr_market_value'] > 0,
                     gdf['assr_land_value'] / gdf['assr_market_value'].clip(lower=1), 1.0)
gdf['re_land'] = gdf['avm_value'] * land_frac
gdf['re_impr'] = gdf['avm_value'] * (1 - land_frac)

tx = gdf[gdf['full_exmp'] == 0].copy()
_lm, _im, _rev, tx = model_split_rate_tax(
    tx, 're_land', 're_impr', current_revenue=tx['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)
gdf['lvt_tax'] = 0.0
gdf.loc[tx.index, 'lvt_tax'] = tx['new_tax'].values

gdf = decompose_reassessment_and_lvt(
    gdf, current_tax_col='current_tax', reassessed_tax_col='avm_flat_tax', final_tax_col='lvt_tax',
)
# Identity holds exactly
assert np.allclose(gdf['reassess_change'] + gdf['lvt_change'], gdf['total_change'])
print('reassess_change + lvt_change == total_change  ✓')

In [ ]:
# Median % change at each stage, by class code — the two effects side by side.
ne = gdf[gdf['full_exmp'] == 0]
dec = ne.groupby('category_code').agg(
    reassess=('reassess_change_pct', 'median'),
    lvt=('lvt_change_pct', 'median'),
).sort_values('lvt')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
dec['reassess'].plot.barh(ax=ax1, color=['#d62728' if v > 0 else '#2ca02c' for v in dec['reassess']])
ax1.axvline(0, color='black', lw=0.8); ax1.set_title('Reassessment effect\n(CAMA → AVM flat)')
ax1.set_xlabel('Median % change')
dec['lvt'].plot.barh(ax=ax2, color=['#d62728' if v > 0 else '#2ca02c' for v in dec['lvt']])
ax2.axvline(0, color='black', lw=0.8); ax2.set_title('LVT effect\n(AVM flat → split-rate)')
ax2.set_xlabel('Median % change')
plt.suptitle('Reading — reassessment vs LVT decomposition (demo split)')
plt.tight_layout()
plt.savefig(DATA_DIR / 'reassessment_demo_decomposition.png', dpi=130)
plt.close()
print('Saved data/reassessment_demo_decomposition.png')
dec.round(1)

**For income/minority quintile views** of these effects, join Census demographics
(`lvt.census_utils.match_to_census_blockgroups`) and pass the two change columns to
`lvt.viz.quintile_progressivity_chart`, e.g.:

```python
quintile_progressivity_chart(
    gdf, quintile_col='median_income',
    value_cols=[('reassess_change', 'Reassessment', '#d62728'),
                ('lvt_change', 'LVT reform', '#2ca02c')],
    flip_sign_for=['reassess_change', 'lvt_change'],
)
```